# 99_explore — optional read-only exploration support

Use this optional support notebook for discovery, profiling, troubleshooting, investigation, and ad hoc analysis. The required delivery path remains: `01_agreement` → `02_pipeline` → `03_governance`.

`99_explore` can read selected agreement context and existing catalogue context to help you investigate a source table. It does **not** approve agreements, enforce guardrails, write pipeline metadata, register delivery state, promote outputs, or mutate governance metadata.

Keep repeatable transformation logic in `02_pipeline`. Keep approval and review workflows in `01_agreement` or `03_governance`.


# Maintainer Notes

> **Minimum required FabricOps release:** v0.2.0

## FabricOps compatibility

| Tested FabricOps release | Environment | Tested by | Date tested |
|---|---|---|---|
| v0.2.0 | Microsoft Fabric | Voyce | 13 Jul 2026 |

## 01 Configure environment


In [ ]:
%run 00_env_config


## 02 Import functions


In [ ]:
from fabricops_kit import (
   
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
    
    profile_dataframe,
    widget_browse_metadata_catalogue,
    widget_pipeline_bootstrap,
)


## 03 Read from user-defined source

Pick one source read example and set its `RUN_*` flag to `True`.

For Lakehouse file helpers, paths are relative to the configured Lakehouse **Files** area. For example, `input/orders.csv` resolves under `Files/input/orders.csv`; do not add a leading `Files/` prefix in these helper calls.


In [ ]:
# ============================================================
# Shared exploration inputs
# ============================================================

source_df = None
source_table_name = "orders_csv"
source_target = "source"
source_schema = SOURCE_SCHEMA


In [ ]:
# Example CSV file from the Source lakehouse.
# Set RUN_CSV_SOURCE = True after updating the path/table name.

RUN_CSV_SOURCE = False

if RUN_CSV_SOURCE:
    source_table_name = "orders_csv"

    source_df = read_lakehouse_csv(
        "orders.csv",
        target=source_target,
        spark_session=spark,
        header=True,
        inferSchema=True,
    )

    display(source_df)


In [ ]:
# Example Excel file from the Source lakehouse.
# Set RUN_EXCEL_SOURCE = True after updating the path, sheet, and table name.

RUN_EXCEL_SOURCE = False

if RUN_EXCEL_SOURCE:
    source_table_name = "products_excel"

    source_df = read_lakehouse_excel(
        "products.xlsx",
        target=source_target,
        sheet_name="products",
        spark_session=spark,
    )

    display(source_df)


In [ ]:
# Example Parquet file from the Source lakehouse.
# Set RUN_PARQUET_SOURCE = True after updating the path/table name.

RUN_PARQUET_SOURCE = False

if RUN_PARQUET_SOURCE:
    source_table_name = "customers_parquet"

    source_df = read_lakehouse_parquet(
        "customers.parquet",
        target=source_target,
        spark_session=spark,
    )

    display(source_df)


In [ ]:
# Example Lakehouse Delta table read from the Source lakehouse.
# Set RUN_LAKEHOUSE_TABLE_SOURCE = True after updating source_table_name/schema/target.

RUN_LAKEHOUSE_TABLE_SOURCE = False

if RUN_LAKEHOUSE_TABLE_SOURCE:
    source_table_name = "your_source_table_name"

    source_df = read_lakehouse_table(
        source_table_name,
        target=source_target,
        schema=source_schema,
        spark_session=spark,
    )

    source_df.printSchema()
    display(source_df.limit(20))


### Optional write/read smoke tests

These smoke tests are disabled by default because they write temporary tables. Enable only when you intentionally want to test FabricOps write/read routing for the configured target.


In [ ]:
# Optional smoke test — write current source_df to Unified lakehouse, then read it back.

RUN_LAKEHOUSE_SMOKE_TEST = False

if RUN_LAKEHOUSE_SMOKE_TEST:
    if source_df is None:
        raise ValueError("Load a source_df before running the Lakehouse smoke test.")

    write_lakehouse_table(
        source_df,
        table_name="smoke_test_source_df",
        target="unified",
        mode="overwrite",
        options={"overwriteSchema": "true"},
    )

    lakehouse_df = read_lakehouse_table(
        table_name="smoke_test_source_df",
        target="unified",
        spark_session=spark,
    )

    display(lakehouse_df)


In [ ]:
# Optional smoke test — write current source_df to Product warehouse, then read it back.
# Warehouse table reads load the full schema.table, so reserve them for small tables, lookups,
# smoke tests, or intentional full-table reads. Prefer read_warehouse_query for filtered reads.

RUN_WAREHOUSE_SMOKE_TEST = False

if RUN_WAREHOUSE_SMOKE_TEST:
    if source_df is None:
        raise ValueError("Load a source_df before running the Warehouse smoke test.")

    write_warehouse_table(
        source_df,
        schema="dbo",
        table_name="smoke_test_source_df",
        target="product",
        mode="overwrite",
    )

    warehouse_df = read_warehouse_table(
        schema="dbo",
        table_name="smoke_test_source_df",
        target="product",
        spark_session=spark,
    )

    display(warehouse_df)

    warehouse_query_df = read_warehouse_query(
        "SELECT TOP 100 * FROM dbo.smoke_test_source_df",
        target="product",
        spark_session=spark,
    )

    display(warehouse_query_df)


## 04 Optional standardized data profiling of your defined source

This profile is local exploratory output only. It does not update metadata tables, create catalogue evidence, approve governance, or enforce guardrails.

It lists each column in your DataFrame, its data type, total row count, null count, distinct count, and simple min/max evidence where applicable.


In [ ]:
RUN_PROFILE = False

if RUN_PROFILE:
    if source_df is None:
        raise ValueError("Load a source_df before running local profiling.")

    display(
        profile_dataframe(
            source_df,
            table_name=source_table_name,
        )
    )


## 05 Browse metadata catalogue

Use the searchable FabricOps catalogue browser to select a configured logical FabricStore target, then browse tables recorded under that target.


In [ ]:
catalogue_browser = widget_browse_metadata_catalogue(
    agreement=AGREEMENT,
    target="metadata",
    schema=METADATA_SCHEMA,
    spark_session=spark,
)

# After changing the widget selectors, rerun this cell or the line below to fetch
# the current filtered Spark DataFrame.
latest_catalogue = catalogue_browser["get_dataframe"]()
latest_catalogue


## Optional -Select agreement

Tie this back to a data agreement for govenance visiblity

In [ ]:
PIPELINE = widget_pipeline_bootstrap(
    notebook_type="99_explore",
    select_agreement=True,
    register_notebook=False,
    read_only=True,
)

AGREEMENT = PIPELINE.agreement
AGREEMENT
